# Optimizers — How Weights Get Updated

After backprop computes gradients, the optimizer decides **how to update weights**.
All optimizers implement the core idea: `W_new = W_old - lr * f(gradient)`
The difference is what `f(gradient)` does.

**Key optimizers in order of evolution:**
```
SGD → SGD + Momentum → RMSProp → Adam → AdamW
```

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

def make_model_and_data():
    model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
    X = torch.randn(100, 4)
    y = X[:, 0] * 2 - X[:, 1] + 0.5 + 0.1 * torch.randn(100)
    return model, X, y

loss_fn = nn.MSELoss()

## SGD — Stochastic Gradient Descent

**Math:** `W = W - lr * gradient`

Simplest possible update. Sensitive to learning rate. No memory of past gradients.

In [ ]:
model, X, y = make_model_and_data()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for step in range(5):
    pred = model(X).squeeze()
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'SGD step {step+1} | loss: {loss.item():.4f}')

## SGD + Momentum

**Math:** `velocity = β * velocity + gradient`  then  `W = W - lr * velocity`

Keeps a running average of past gradients (momentum). Smooths out noisy gradients,
accelerates in consistent directions. `β=0.9` means 90% old velocity, 10% new gradient.

In [ ]:
model, X, y = make_model_and_data()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

for step in range(5):
    pred = model(X).squeeze()
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'SGD+momentum step {step+1} | loss: {loss.item():.4f}')

## Adam — Adaptive Moment Estimation

**Math (simplified):**
```
m = β1 * m + (1-β1) * gradient          ← 1st moment: running mean of gradients
v = β2 * v + (1-β2) * gradient²         ← 2nd moment: running mean of gradient²
W = W - lr * m / (sqrt(v) + ε)          ← adaptive lr per parameter
```

Each parameter gets its own effective learning rate based on its gradient history.
Parameters with large gradients get smaller steps; sparse ones get larger steps.

**Default hyperparameters:** `lr=1e-3, β1=0.9, β2=0.999, ε=1e-8`

In [ ]:
model, X, y = make_model_and_data()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for step in range(5):
    pred = model(X).squeeze()
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'Adam step {step+1} | loss: {loss.item():.4f}')

## AdamW — Adam + Weight Decay (standard for LLMs)

**Problem with Adam:** the original Adam applies weight decay incorrectly (mixes it into the adaptive gradient scaling).

**AdamW fix:** applies weight decay directly to weights, separately from the gradient update:
```
W = W - lr * m / (sqrt(v) + ε)   ← gradient step (same as Adam)
W = W * (1 - lr * weight_decay)  ← weight decay step (independent)
```

**This is the standard optimizer for LLM training** (GPT, LLaMA, BERT all use AdamW).

In [ ]:
model, X, y = make_model_and_data()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01    # L2 regularization — penalizes large weights
)

for step in range(5):
    pred = model(X).squeeze()
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'AdamW step {step+1} | loss: {loss.item():.4f}')

## Learning Rate Schedulers

Learning rate is not constant during training. Schedulers decay it over time:
- High lr at start → explore
- Low lr later → fine-tune

In [ ]:
model, X, y = make_model_and_data()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# CosineAnnealingLR — smoothly decays lr following a cosine curve
# most common for LLM training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

for step in range(10):
    pred = model(X).squeeze()
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()   # update lr after each step
    print(f'step {step+1:2d} | loss: {loss.item():.4f} | lr: {scheduler.get_last_lr()[0]:.6f}')

# Linear warmup + cosine decay — used in LLM training:
# scheduler = torch.optim.lr_scheduler.OneCycleLR(
#     optimizer, max_lr=1e-3, steps_per_epoch=100, epochs=10
# )

## Optimizer cheat sheet

| Optimizer | Use case |
|---|---|
| SGD | Computer vision (CNNs) — often outperforms Adam with right lr |
| SGD + Momentum | Same, better convergence |
| Adam | General purpose, good default |
| **AdamW** | **LLMs, Transformers — the standard** |
| Adagrad | Sparse data (NLP bag-of-words, recommender systems) |